# torch.distributed

- torch.distributed 是 PyTorch 中用于分布式训练的核心模块，它提供了非常丰富的 API。为了让你更直观地理解，我们可以将这些方法按功能划分为四大类：
    - 环境初始化与进程管理、
    - 集合通信原语、
    - 点对点通信以及
    - 分布式数据加载
以下是 torch.distributed 中最核心和常用的方法分类介绍：

## 🛠️ 1. 环境初始化与进程管理

这是启动任何分布式任务的第一步，用于配置通信后端和获取当前进程的身份信息。
- <font color ='red'>init_process_group(backend, init_method, ...)</font>：初始化分布式训练环境。你需要指定通信后端（如 GPU 常用的 'nccl' 或 CPU 可用的 'gloo'）以及初始化方式（如 'env://' 通过环境变量配置）。
- <font color ='red'>get_rank()：</font>获取当前进程的全局唯一编号（Rank ID），通常从 0 开始。
- <font color ='red'>get_world_size()：</font>获取全局进程的总数（即总共用了多少张卡或多少个进程）。
- <font color ='red'>get_local_rank()：</font>获取当前进程在当前节点（机器）内的编号（通常对应本机的 GPU 卡号）。
- <font color ='red'>barrier(group)：</font>进程同步屏障。执行到这行代码时，所有进程会阻塞等待，直到所有进程都到达该点后，才会一起继续向下执行。常用于确保各进程步调一致。
- <font color ='red'>destroy_process_group()</font>：销毁进程组，释放分布式环境占用的资源，通常在程序结束时调用。

## 📡 2. 集合通信原语 (Collective Communication)

这是分布式训练中最核心的部分，用于在多个进程之间高效地搬运和计算张量数据。你之前问到的 all_gather 就属于这一类。
- <font color ='red'>all_reduce(tensor, op, group)</font>：全归约。将所有进程的数据按指定操作（如求和 SUM、取平均 AVG）进行合并，并将结果广播给所有进程。常用于 DDP 训练中的梯度同步。
- <font color ='red'>all_gather(tensor_list, tensor, group)</font>：全收集。收集所有进程的张量，并分发给所有进程。
- <font color ='red'>gather(tensor, gather_list, dst, group)</font>：收集。将所有进程的张量收集到指定的目标进程（dst）上，其他进程不保留完整结果。
- <font color ='red'>reduce(tensor, dst, op, group)</font>：归约。将所有进程的数据按指定操作合并，并将最终结果只保留在指定的目标进程（dst）上。
- <font color ='red'>broadcast(tensor, src, group)</font>：广播。将源进程（src）的张量数据复制并发送给组内的所有其他进程。
- <font color ='red'>scatter(tensor, scatter_list, src, group)</font>：分发。源进程（src）将一个列表中的多个张量，分别发送给组内的各个进程。
- <font color ='red'>reduce_scatter(output, input_list, op, group)</font>：归约分发。先对各个进程的张量列表进行归约求和，然后将结果切分并分散到各个进程中。
(注：以上集合通信方法通常都有对应的异步版本，即在方法名前加 i，如 ireduce, ibroadcast 等，它们会立即返回一个句柄，允许计算和通信重叠)

## 🔄 3. 点对点通信 (Point-to-Point Communication)
用于在两个指定的进程之间直接发送和接收数据，灵活性高但编程复杂度也更高。
- <font color ='red'>send(tensor, dst, group)：</font>阻塞式发送。将张量发送给指定的目标进程（dst）。
- <font color ='red'>recv(tensor, src, group)：</font>阻塞式接收。从指定的源进程（src）接收张量。
- <font color ='red'>isend(tensor, dst, group)：</font>非阻塞式发送。立即返回，不等待发送完成。
- <font color ='red'>irecv(tensor, src, group)：</font>非阻塞式接收。立即返回，不等待接收完成。

## 📦 4. 分布式数据加载与工具
- <font color ='red'>DistributedSampler</font>：这是一个位于 torch.utils.data.distributed 下的采样器，但在分布式训练中必不可少。它的作用是自动将数据集切分，确保每个进程（Rank）在每个 Epoch 只读取属于自己那一部分的数据，避免数据重复。
- <font color ='red'>all_gather_object / broadcast_object_list</font>：专门用于收集或广播普通的 Python 对象（如字典、列表、字符串等），底层通过 pickle 序列化实现。

## 💡 总结
- 对于绝大多数大模型训练和微调任务，你最常打交道的通常是 环境初始化类（init_process_group, get_rank）和 集合通信类（all_reduce, all_gather）。其他的 API 则更多用于实现一些高度定制化的分布式算法或底层通信优化。